# MLOps Assignment 2 — Goodreads Genre Classification

**Fine-tuning DistilBERT on UCSD Goodreads reviews — tracked with W&B, published to Hugging Face Hub.**

- **Author:** SIDDHANT SINGH (g25ait2105) · PGD AI · IIT Jodhpur
- **Platform:** Kaggle Notebook (GPU T4 x2)
- **Tracking:** Weights & Biases
- **Model registry:** Hugging Face Hub

This notebook is the single executable deliverable. It loads UCSD Goodreads reviews for 8 genres, fine-tunes `distilbert-base-cased`, logs every metric to W&B, saves the classification report as a W&B Artifact, and finally pushes the trained model + tokenizer to a public HF repo.


## 0. Kaggle Setup (one-time per notebook)

**Before running anything below**, in the Kaggle UI:
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On** (needed for dataset download + HF/W&B pushes)
3. **Add-ons → Secrets → Add:**
   - `WANDB_API_KEY` — get from https://wandb.ai/settings
   - `HF_TOKEN` — get from https://huggingface.co/settings/tokens (needs **Write** permission)
4. Check both secrets are **Attached to notebook**.


## 1. Load API tokens from Kaggle Secrets

In [1]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN']      = secrets.get_secret('HF_TOKEN')

print("Secrets loaded.")


Secrets loaded.


## 2. Install / upgrade libraries

In [2]:
!pip install -q -U transformers wandb huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 68.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


## 3. Imports & parameters

In [3]:
from collections import defaultdict
import random, json, gzip, pickle, os
import numpy as np
import pandas as pd
import requests
import torch

from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import wandb
from huggingface_hub import login as hf_login

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

model_name   = 'distilbert-base-cased'
device_name  = 'cuda' if torch.cuda.is_available() else 'cpu'
max_length   = 512
HF_REPO_ID   = 'G25AIT2105/MLops'
WANDB_PROJECT = 'mlops-assignment2'
RUN_NAME     = 'distilbert-goodreads-run-1'

print('Device:', device_name)
print('Model :', model_name)


Device: cuda
Model : distilbert-base-cased


## 4. Load & sample Goodreads reviews per genre

In [4]:
genre_url_dict = {
    'poetry':                 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz',
    'children':               'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz',
    'comics_graphic':         'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz',
    'fantasy_paranormal':     'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz',
    'history_biography':      'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz',
    'mystery_thriller_crime': 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz',
    'romance':                'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz',
    'young_adult':            'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz',
}


def load_reviews(url, head=10000, sample_size=2000):
    """Stream a gzipped JSON of reviews and return a random sample of review_text strings."""
    reviews = []
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with gzip.open(response.raw, 'rt', encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            reviews.append(json.loads(line)['review_text'])
            if i + 1 >= head:
                break
    return random.sample(reviews, min(sample_size, len(reviews)))


genre_reviews_dict = {}
for genre, url in genre_url_dict.items():
    print(f'Loading {genre} ...')
    genre_reviews_dict[genre] = load_reviews(url, head=10000, sample_size=2000)
    print(f'  -> {len(genre_reviews_dict[genre])} reviews')

# Persist locally for fallback / re-runs
with open('genre_reviews_dict.pickle', 'wb') as fh:
    pickle.dump(genre_reviews_dict, fh)


Loading poetry ...
  -> 2000 reviews
Loading children ...
  -> 2000 reviews
Loading comics_graphic ...
  -> 2000 reviews
Loading fantasy_paranormal ...
  -> 2000 reviews
Loading history_biography ...
  -> 2000 reviews
Loading mystery_thriller_crime ...
  -> 2000 reviews
Loading romance ...
  -> 2000 reviews
Loading young_adult ...
  -> 2000 reviews


### Fallback if automatic download fails

If any of the URLs above fail on Kaggle, download the `.json.gz` files manually from
<https://mengtingwan.github.io/data/goodreads.html#datasets>, upload them as a Kaggle **Dataset**,
attach it to this notebook, and replace `requests.get(url, stream=True)` with a local file open
on `/kaggle/input/<your-dataset>/...`.


## 5. Train / test split (800 train + 200 test per genre)

In [5]:
train_texts, train_labels = [], []
test_texts,  test_labels  = [], []

for genre, reviews in genre_reviews_dict.items():
    sampled = random.sample(reviews, 1000)
    for r in sampled[:800]:
        train_texts.append(r); train_labels.append(genre)
    for r in sampled[800:]:
        test_texts.append(r);  test_labels.append(genre)

print('Train:', len(train_texts), '| Test:', len(test_texts))
print('Genres:', sorted(set(train_labels)))


Train: 6400 | Test: 1600
Genres: ['children', 'comics_graphic', 'fantasy_paranormal', 'history_biography', 'mystery_thriller_crime', 'poetry', 'romance', 'young_adult']


## 6. Tokenize + build label maps + encode

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

unique_labels = sorted(set(train_labels))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(unique_labels)
print('Labels:', label2id)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=max_length)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=max_length)

train_labels_encoded = [label2id[y] for y in train_labels]
test_labels_encoded  = [label2id[y] for y in test_labels]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Labels: {'children': 0, 'comics_graphic': 1, 'fantasy_paranormal': 2, 'history_biography': 3, 'mystery_thriller_crime': 4, 'poetry': 5, 'romance': 6, 'young_adult': 7}


## 7. Custom Torch dataset

In [7]:
class GoodreadsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = GoodreadsDataset(train_encodings, train_labels_encoded)
test_dataset  = GoodreadsDataset(test_encodings,  test_labels_encoded)


## 8. Load the pre-trained DistilBERT model

In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(device_name)


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 9. Initialise W&B run + define compute_metrics

In [9]:
wandb.login(key=os.environ['WANDB_API_KEY'])

wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        'model':         model_name,
        'epochs':        3,
        'batch_size':    16,
        'learning_rate': 3e-5,
        'max_length':    max_length,
        'dataset':       'UCSD Goodreads',
        'num_labels':    NUM_LABELS,
        'platform':      'Kaggle',
    },
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: g25ait2105 (g25ait2105-indian-institute-of-technology-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 10. Training arguments + Trainer

In [10]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='wandb',
    run_name=RUN_NAME,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


## 11. Train

In [11]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.617096,2.511100,0.560000,0.555023
2,2.112182,2.272588,0.608750,0.609581
3,1.596670,2.278800,0.608750,0.609612


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=2.3941350173950195, metrics={'train_runtime': 510.342, 'train_samples_per_second': 37.622, 'train_steps_per_second': 1.176, 'total_flos': 2543646198988800.0, 'train_loss': 2.3941350173950195, 'epoch': 3.0})

## 12. Final evaluation + log final metrics to W&B

In [12]:
eval_results = trainer.evaluate()
print(eval_results)

wandb.log({
    'final/loss':     eval_results['eval_loss'],
    'final/accuracy': eval_results['eval_accuracy'],
    'final/f1':       eval_results['eval_f1'],
})


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1
1.596670,2.278800,3,0.608750,0.609612


{'eval_loss': 2.2787997722625732, 'eval_accuracy': 0.60875, 'eval_f1': 0.6096122478246423}


## 13. Classification report → save as file → upload as W&B Artifact

In [13]:
preds = trainer.predict(test_dataset).predictions.argmax(-1)
labels = [item['labels'].item() for item in test_dataset]

report = classification_report(
    labels, preds,
    target_names=[id2label[i] for i in range(NUM_LABELS)],
    output_dict=True,
)

with open('eval_report.json', 'w') as fh:
    json.dump(report, fh, indent=2)

# Also save a human-readable text version
with open('eval_report.txt', 'w') as fh:
    fh.write(classification_report(
        labels, preds,
        target_names=[id2label[i] for i in range(NUM_LABELS)],
    ))

artifact = wandb.Artifact('eval-report', type='evaluation')
artifact.add_file('eval_report.json')
artifact.add_file('eval_report.txt')
wandb.log_artifact(artifact)

print(open('eval_report.txt').read())


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                        precision    recall  f1-score   support

              children       0.70      0.69      0.69       200
        comics_graphic       0.84      0.73      0.78       200
    fantasy_paranormal       0.41      0.48      0.44       200
     history_biography       0.61      0.53      0.56       200
mystery_thriller_crime       0.59      0.69      0.63       200
                poetry       0.75      0.81      0.78       200
               romance       0.61      0.55      0.58       200
           young_adult       0.42      0.40      0.41       200

              accuracy                           0.61      1600
             macro avg       0.61      0.61      0.61      1600
          weighted avg       0.61      0.61      0.61      1600



## 14. Push trained model + tokenizer to Hugging Face Hub

In [14]:
hf_login(token=os.environ['HF_TOKEN'])

model.push_to_hub(HF_REPO_ID, private=False)
tokenizer.push_to_hub(HF_REPO_ID, private=False)

HF_URL = f'https://huggingface.co/{HF_REPO_ID}'
wandb.run.summary['huggingface_model'] = HF_URL
wandb.run.summary['final_accuracy']    = eval_results['eval_accuracy']
wandb.run.summary['final_f1']          = eval_results['eval_f1']
wandb.run.summary['final_loss']        = eval_results['eval_loss']

print('Pushed to:', HF_URL)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to: https://huggingface.co/G25AIT2105/MLops


## 15. Close the W&B run

In [15]:
wandb.finish()


eval/accuracy,▁███
eval/f1,▁███
eval/loss,█▁▁▁
eval/runtime,█▁▂▁
eval/samples_per_second,▁█▇█
eval/steps_per_second,▁▇▆█
final/accuracy,▁
final/f1,▁
final/loss,▁
test/accuracy,▁
+10,...


## Done

After this notebook finishes:
- The W&B run page (under project `mlops-assignment2`) shows training curves + final metrics + the HF model URL.
- The HF repo `G25AIT2105/MLops` is public and contains the trained weights and tokenizer.
- The classification report is attached to the W&B run as an Artifact named `eval-report`.

Take a screenshot of the W&B run charts for the report.
